# Your recommender scores well. Who is it good *for*?

Over Units 2-10 we pushed hit@10 from 0.108 (popularity) to **0.362** (Unit 10's hybrid). That is a
real gain on our scoreboard. But a scoreboard answers only one question -- *did a book the reader later read
land in their top 10?* -- and a recommender that ships does three more things the scoreboard never sees:

1. **It decides which books get seen at all.** Every top-10 slot is a scarce piece of shelf space. If the
   most accurate path hands nearly every slot to the same few hundred bestsellers, most of the catalog --
   and most of its authors -- is never shown to anyone.
2. **It changes what readers read next.** Readers pick from what they are shown, those reads land in the
   next training log, and the model is retrained on them. Does that loop *spread* reading out, or
   *concentrate* it?
3. **It shapes each reader's view of the catalog.** Does a reader who reads widely get a top 10 that is
   just as wide -- or a narrower slice (a *filter bubble*)?

This unit **measures** all three on our data. Measured is the operative word: every claim below comes
with a number, every "fairness" number is a *descriptive* proxy (not a verdict), and where two things
we want pull against each other we draw the **trade-off curve** instead of declaring a winner. Who should
pick the point on that curve is a question for people, not for the code -- but the code can tell them
exactly what each point costs.

## 0. Setup: one cohort, one rule

Everything in this unit is read off the **same lists**: for each of the **500 warm val readers** (readers
with a `val` book they have not already read in train; cold readers excluded -- the scoreboard's own
cohort since Unit 1), the top 10 books a path recommends, with their train books excluded. `val_cohort`
is that cohort rule as a helper; we check it against the rule written out by hand.

We also build each reader's **held-out set**: every positive *outside* train (`val`, and the sealed
final split). We never look at what is in the sealed part -- we only need the set so that, in Section 3,
the feedback-loop simulation can promise never to write a held-out answer into the training log.

In [1]:
import json
import math
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from bookrec import (
    FeatureTowerRetrievalPath,
    HybridRetrievalPath,
    ItemItemRetrievalPath,
    LexicalRetrievalPath,
    MatrixFactorizationPath,
    PopularityRetrievalPath,
    SemanticEmbeddingRetrievalPath,
    TwoTowerRetrievalPath,
    author_catalog_share,
    author_exposure_gini,
    blend,
    calibrate_scores,
    catalog_coverage,
    exposure_counts,
    exposure_gini,
    generated_dir,
    genre_entropy,
    genre_spread_vs_history,
    gini,
    head_ids_from_counts,
    head_share,
    load_catalog,
    load_glove_subset,
    load_keywords,
    long_tail_share,
    popularity_penalized,
    rank,
    run_validation_scoreboard,
    simulate_feedback_loop,
    val_cohort,
)

K = 10
SEED = 0

data_dir = generated_dir()
catalog = load_catalog(data_dir / "catalog.csv.gz")
catalog_ids = sorted(catalog)
interactions_path = data_dir / "interactions.csv.gz"
rows = pd.read_csv(interactions_path).to_dict("records")
keywords = load_keywords(data_dir / "keywords.csv.gz")
glove = load_glove_subset()
cold_readers = set(json.loads((data_dir / "cold_partitions.json").read_text())["cold_readers"])

seen_by = defaultdict(set)        # train positives: what each reader has already read
val_by = defaultdict(set)         # val positives: the scoreboard's targets
held_out_by = defaultdict(set)    # every positive OUTSIDE train: the feedback loop's no-write list
for row in rows:
    if int(row["label"]) != 1:
        continue
    reader_id, item_id = int(row["reader_id"]), int(row["item_id"])
    if row["split"] == "train":
        seen_by[reader_id].add(item_id)
    else:
        held_out_by[reader_id].add(item_id)
        if row["split"] == "val":
            val_by[reader_id].add(item_id)

cohort = val_cohort(rows, catalog_ids, cold_readers)
assert cohort == [r for r in sorted(val_by) if r not in cold_readers and val_by[r] - seen_by[r]]
print(f"catalog {len(catalog_ids)} books; {len(rows)} interaction rows")
print(f"val cohort: {len(cohort)} warm readers (the same 500 every number below is read on)")

catalog 2000 books; 27004 interaction rows
val cohort: 500 warm readers (the same 500 every number below is read on)


## 1. The whole-system scoreboard

We bring back every path from Units 2-10 -- popularity (U2), BM25 lexical (U3), item-item CF (U4),
matrix factorization (U5), Unit 6's four-way blend, semantic embeddings (U7), the two-tower (U8), the
feature tower (U9) and Unit 10's hybrid -- and read **three** numbers off each one's top-10 lists:

- **hit@10** -- the accuracy we have chased all book;
- **coverage** (Unit 2) -- the fraction of the 2,000-book catalog that appears in *anyone's* top 10;
- **head-share** (Unit 2) -- the fraction of all top-10 *slots* that go to the **head**: the 10% of
  books (200) with the most train reads.

The cheap paths fit in seconds. The three learned paths are the unit's only expensive cells, and each is
fit **once** and reused by every section below.

In [2]:
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
lexical = LexicalRetrievalPath(keywords).fit(rows, catalog=catalog_ids)
semantic = SemanticEmbeddingRetrievalPath(keywords, glove).fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
mf = MatrixFactorizationPath(seed=SEED).fit(rows, catalog=catalog_ids)
print("fitted: popularity, lexical (BM25), semantic, item-item CF, matrix factorization")

fitted: popularity, lexical (BM25), semantic, item-item CF, matrix factorization


In [3]:
two_tower = TwoTowerRetrievalPath(seed=SEED).fit(rows, catalog=catalog_ids)
print("fitted: two-tower (Unit 8, seed 0)")

fitted: two-tower (Unit 8, seed 0)


In [4]:
feature_tower = FeatureTowerRetrievalPath(  # Unit 9's shipped default: warm-only negatives
    catalog, keywords, glove, n_epochs=40, negative_pool="warm", seed=SEED
).fit(rows, catalog=catalog_ids)
print("fitted: feature tower (Unit 9, warm-only negatives, seed 0)")

fitted: feature tower (Unit 9, warm-only negatives, seed 0)


Two composite paths need no new fit. Unit 10's **hybrid** fuses BM25 and the two-tower (dense weight 0.7,
pool 50). Unit 6's **four-way blend** has no path class of its own -- Unit 6 scored it with
`run_blended_scoreboard` -- so we wrap the same recipe (each member retrieves a pool of 30;
`blend` calibrates and takes the weighted union) in a tiny class with a `retrieve` method. That lets
every tool in this unit treat it like any other path.

`top10(path)` applies the scoreboard's rule to one path and keeps every reader's list. Its mean hit must
reproduce `run_validation_scoreboard` -- we check it on item-item.

In [5]:
hybrid = HybridRetrievalPath(lexical, two_tower, weight=0.7, pool=50).fit(rows)


class BlendPath:
    # Unit 6's blend as a retrieval path: each member retrieves `pool`, blend() merges them.
    name = "u6-blend"

    def __init__(self, members, weights, pool=30):
        self.members, self.weights, self.pool = members, weights, pool

    def retrieve(self, query, context, k):
        per_path = {n: p.retrieve(query, context, self.pool) for n, p in self.members.items()}
        return blend(per_path, weights=self.weights)[:k]


u6_blend = BlendPath(
    {"popularity": popularity, "lexical": lexical, "item-item": item_item, "mf": mf},
    {"popularity": 0.25, "lexical": 0.5, "item-item": 1.0, "mf": 1.0},
)

paths = {
    "popularity": popularity, "lexical": lexical, "semantic": semantic, "item-item": item_item,
    "MF": mf, "two-tower": two_tower, "feature-tower": feature_tower,
    "U10 hybrid": hybrid, "U6 blend": u6_blend,
}


def top10(path, pool=K):
    # Each cohort reader's top 10 by the scoreboard's rule: seen excluded, rank keeps 10.
    return {
        r: [c.item_id for c in rank(path.retrieve(r, {"seen": seen_by[r]}, pool), n=K,
                                    exclude=seen_by[r])]
        for r in cohort
    }


def hit_at_10(tops):
    # Mean per-reader hit: 1.0 if any unseen val book is in the reader's top 10.
    return float(np.mean([float(bool(set(tops[r]) & (val_by[r] - seen_by[r]))) for r in cohort]))


tops = {name: top10(path) for name, path in paths.items()}
board = run_validation_scoreboard(item_item, interactions_path, catalog_ids=catalog_ids, k=K,
                                  cold_readers=cold_readers)
assert abs(hit_at_10(tops["item-item"]) - board.hit_rate_at_k) < 1e-12

head = set(head_ids_from_counts(popularity.counts, catalog_ids))   # top 10% by train reads
print(f"head = {len(head)} most-read books (10% of the catalog)\n")
print(f"{'path':<14} {'hit@10':>7} {'coverage':>9} {'head-share':>11}")
for name, lists in tops.items():
    print(f"{name:<14} {hit_at_10(lists):>7.3f} {catalog_coverage(lists.values(), catalog_ids):>9.3f} "
          f"{head_share(lists.values(), head):>11.3f}")

head = 200 most-read books (10% of the catalog)

path            hit@10  coverage  head-share
popularity       0.108     0.008       1.000
lexical          0.158     0.278       0.268
semantic         0.102     0.354       0.109
item-item        0.252     0.490       0.491
MF               0.276     0.074       0.994
two-tower        0.340     0.177       0.900
feature-tower    0.298     0.306       0.638
U10 hybrid       0.362     0.192       0.862
U6 blend         0.306     0.333       0.793


Read the table across, not down. The **most accurate** paths are the **most concentrated**:

- **MF** (hit about 0.276) puts about 99% of its slots on the 200 head books and shows only about 7% of
  the catalog to anyone. The **two-tower** (0.340) and the **U10 hybrid** (0.362, the best hit) give
  about 90% and 86% of their slots to the head and cover under a fifth of the catalog.
- **Item-item CF** is the *least* concentrated collaborative path: under half its slots go to the head
  and it covers about 49% of the catalog -- at a lower hit (0.252).
- **Popularity** is the extreme: 100% head, 0.8% coverage -- every reader gets nearly the same list.
- The content paths (lexical, semantic) cover a lot and lean on the tail, but at low hit.

Hit@10 alone would have told you to ship the hybrid and stop. Coverage and head-share say the
hybrid's accuracy comes with a very narrow shelf. Neither number is "the" answer; together they are a
**whole-system** view. The next section makes "narrow" precise.

## 2. Exposure fairness: how unequal is the shelf?

Coverage counts which books are shown *at all*; head-share uses one cut-off. To summarise the **whole
distribution** of exposure in one number we borrow the economists' **Gini coefficient**. Give every
catalog book its **exposure** $x_i$ = the number of top-10 slots it received across the 500 readers
(most books get 0). Then

$$G = \frac{\sum_i \sum_j |x_i - x_j|}{2\, n^2\, \bar{x}}$$

-- the average gap between two randomly chosen books' exposure, scaled by twice the mean. $G = 0$ means
every book got exactly the same exposure; one book taking every slot gives $G = (n-1)/n$, almost 1.

Two choices matter, and we fix both for every path: $n$ is the **full catalog** (2,000 books, zeros
included -- a book never shown is the most unequal outcome of all, so it must count), and the lists are
always the same 500 readers' top 10s. Only then are two paths' Ginis comparable. By hand first, on toy
vectors, then on item-item's real exposure:

In [6]:
def gini_by_hand(x):
    # Mean absolute gap over all ordered pairs (i, j), divided by twice the mean.
    x = np.array(x, dtype=float)
    return float(abs(x[:, None] - x[None, :]).mean() / (2 * x.mean()))


for toy in ([5, 5, 5, 5], [2, 4, 6, 8], [0, 0, 0, 20]):
    print(f"Gini of {toy}: {gini_by_hand(toy):.3f}")

exposure = Counter(item for top in tops["item-item"].values() for item in top)
x = [exposure.get(i, 0) for i in catalog_ids]               # every catalog book, zeros included
print(f"\nitem-item: {sum(x)} slots over {len(x)} books; {sum(v == 0 for v in x)} books never shown; "
      f"the most-shown book fills {max(x)} slots")
print(f"item-item exposure Gini (by hand): {gini_by_hand(x):.4f}")

Gini of [5, 5, 5, 5]: 0.000
Gini of [2, 4, 6, 8]: 0.250
Gini of [0, 0, 0, 20]: 0.750

item-item: 4990 slots over 2000 books; 1020 books never shown; the most-shown book fills 61 slots
item-item exposure Gini (by hand): 0.7729


`[0, 0, 0, 20]` gives $3/4 = (n-1)/n$: one item takes everything. Item-item's real exposure Gini is about
**0.773** -- quite unequal, even for the *least* concentrated collaborative path, partly because
about half the catalog is never shown at all.

### Reveal: `exposure_counts`, `exposure_gini` and `gini`

`bookrec.gini(values)` computes the same number from the sorted values in $O(n \log n)$ (our pairwise
version builds a $2000 \times 2000$ table). `exposure_counts(lists, catalog_ids)` is our `x` as a
dictionary -- every catalog book present, zeros included -- and `exposure_gini(lists, catalog_ids)` is
the Gini of it. They match the by-hand numbers:

In [7]:
assert exposure_counts(tops["item-item"], catalog_ids) == dict(zip(catalog_ids, x))
assert abs(gini(x) - gini_by_hand(x)) < 1e-9
assert abs(exposure_gini(tops["item-item"], catalog_ids) - gini_by_hand(x)) < 1e-9
assert abs(gini([0, 0, 0, 20]) - 0.75) < 1e-12
print("exposure_counts == by-hand x;  gini == exposure_gini == by-hand pairwise Gini")

exposure_counts == by-hand x;  gini == exposure_gini == by-hand pairwise Gini


### Books are not the only thing that gets exposure: authors

A reader-facing system also distributes attention across **authors**. `author_exposure_gini` sums each
author's slots and takes the Gini over every catalog author. But be careful what you compare it to:
authors do not have equal numbers of books, so even a recommender that showed **every book equally**
would give prolific authors more slots. `author_catalog_share(catalog)` gives each author's share of the
catalog's books; its own Gini is the inequality that comes from the catalog itself, before any
recommender acts. Read every author Gini against that number.

We also add `long_tail_share` -- the fraction of slots that go to non-head books (exactly
$1 - \text{head-share}$).

In [8]:
share = author_catalog_share(catalog)
catalog_author_gini = gini(share.values())
print(f"{len(share)} authors; the largest owns {max(share.values()):.2%} of the catalog")
print(f"author Gini of the CATALOG itself (equal exposure per book): {catalog_author_gini:.3f}\n")

print(f"{'path':<14} {'hit@10':>7} {'item Gini':>10} {'long-tail':>10} {'author Gini':>12}")
for name, lists in tops.items():
    tail = long_tail_share(lists, head)
    assert abs(tail - (1 - head_share(lists.values(), head))) < 1e-12
    print(f"{name:<14} {hit_at_10(lists):>7.3f} {exposure_gini(lists, catalog_ids):>10.3f} "
          f"{tail:>10.3f} {author_exposure_gini(lists, catalog):>12.3f}")

300 authors; the largest owns 0.80% of the catalog
author Gini of the CATALOG itself (equal exposure per book): 0.211

path            hit@10  item Gini  long-tail  author Gini
popularity       0.108      0.995      0.000        0.970
lexical          0.158      0.888      0.732        0.600
semantic         0.102      0.857      0.891        0.573
item-item        0.252      0.773      0.509        0.474
MF               0.276      0.971      0.006        0.843
two-tower        0.340      0.929      0.100        0.687


feature-tower    0.298      0.877      0.362        0.609
U10 hybrid       0.362      0.922      0.138        0.677
U6 blend         0.306      0.908      0.207        0.693


The item Gini tells the same story as head-share with more resolution: from about **0.773** (item-item)
through 0.93 (two-tower) and 0.97 (MF) to 0.995 (popularity). The author Ginis run from about 0.47
(item-item) to 0.97 (popularity) -- all far above the catalog's own **0.211**. So the unequal *author*
exposure is mostly the recommenders' doing, not just a reflection of who wrote more books.

What these numbers are **not**: a fairness verdict. A Gini cannot tell you whether the head books *deserve*
their slots (they are the most-read books, after all), and "fair to authors" is only one side --
readers, too, can be well or badly served. These are **descriptive exposure-inequality proxies**: they
tell you *how* concentrated the shelf is, so that a person can decide how concentrated it *should* be.

### Trading accuracy for exposure: a popularity-penalty re-rank

If we want a less concentrated shelf, the simplest lever is a **re-rank**. Ask a path for a bigger pool
(50 candidates), min-max calibrate its scores to $[0, 1]$ (Unit 6), and subtract a popularity penalty:

$$\mathrm{score}'(i) = \mathrm{minmax}(\mathrm{score}(i)) - \lambda \cdot \frac{\log(1 + \mathrm{count}_i)}{\log(1 + \max \mathrm{count})}$$

where $\mathrm{count}_i$ is the book's train reads. $\lambda = 0$ keeps the path's own order; larger
$\lambda$ pushes popular books down. By hand for one reader, then the reveal:

In [9]:
counts = popularity.counts                          # train reads per book
max_log = math.log1p(max(counts.values()))
LAMBDA = 0.3
reader = cohort[0]

pool = calibrate_scores(item_item.retrieve(reader, {"seen": seen_by[reader]}, 50))
rescored = {c.item_id: c.score - LAMBDA * math.log1p(counts.get(c.item_id, 0)) / max_log for c in pool}
by_hand = sorted(rescored, key=lambda i: (-rescored[i], i))[:K]

penalized = popularity_penalized(item_item, LAMBDA, counts, pool=50)
shipped = [c.item_id for c in penalized.retrieve(reader, {"seen": seen_by[reader]}, K)]
assert shipped == by_hand
print(f"reader {reader}, item-item top 10:      {tops['item-item'][reader]}")
print(f"reader {reader}, penalized (lambda 0.3): {by_hand}")
print(f"head books: {sum(i in head for i in tops['item-item'][reader])} -> {sum(i in head for i in by_hand)}")
print("popularity_penalized(item_item, 0.3, counts, pool=50) == the by-hand re-rank")

reader 1, item-item top 10:      [505, 452, 1688, 1743, 1972, 1050, 418, 180, 184, 413]


reader 1, penalized (lambda 0.3): [505, 1688, 452, 1743, 1972, 1050, 418, 610, 473, 180]
head books: 7 -> 5
popularity_penalized(item_item, 0.3, counts, pool=50) == the by-hand re-rank


`popularity_penalized(path, penalty, counts, pool=50)` wraps any path in that re-rank (it is a path itself,
named `"pop-penalized"`). Now sweep a **predeclared** grid, $\lambda \in \{0, 0.1, 0.3, 0.5, 1.0\}$, on two
bases -- item-item CF and the Unit-6 blend -- and record what each step buys (lower Gini) and costs
(lower hit). We fixed the grid *before* looking, so no point on it was picked because it looked good.

In [10]:
LAMBDAS = (0.0, 0.1, 0.3, 0.5, 1.0)


class PoolMemo:
    # Remember each reader's candidate pool, so the five lambdas share ONE inner retrieval per reader.
    def __init__(self, path):
        self.path, self.memo = path, {}

    def retrieve(self, query, context, k):
        if (query, k) not in self.memo:
            self.memo[(query, k)] = self.path.retrieve(query, context, k)
        return self.memo[(query, k)]


print(f"{'base':<10} {'lambda':>6} {'hit@10':>7} {'hit change':>11} {'coverage':>9} {'item Gini':>10}")
for base in ("item-item", "U6 blend"):
    curve, memo = [], PoolMemo(paths[base])
    for lam in LAMBDAS:
        lists = top10(popularity_penalized(memo, lam, counts, pool=50), pool=50)
        curve.append((hit_at_10(lists), catalog_coverage(lists.values(), catalog_ids),
                      exposure_gini(lists, catalog_ids)))
    for lam, (hit, cov, g) in zip(LAMBDAS, curve):
        print(f"{base:<10} {lam:>6.1f} {hit:>7.3f} {hit - curve[0][0]:>+11.3f} {cov:>9.3f} {g:>10.3f}")
    assert all(a[2] > b[2] for a, b in zip(curve, curve[1:]))     # Gini falls at every step
    assert all(a[0] >= b[0] for a, b in zip(curve, curve[1:]))    # hit never rises

base       lambda  hit@10  hit change  coverage  item Gini


item-item     0.0   0.252      +0.000     0.490      0.773
item-item     0.1   0.244      -0.008     0.498      0.763
item-item     0.3   0.224      -0.028     0.505      0.747
item-item     0.5   0.204      -0.048     0.514      0.731
item-item     1.0   0.112      -0.140     0.511      0.713


U6 blend      0.0   0.306      +0.000     0.333      0.908
U6 blend      0.1   0.304      -0.002     0.365      0.891
U6 blend      0.3   0.290      -0.016     0.414      0.858
U6 blend      0.5   0.262      -0.044     0.450      0.827
U6 blend      1.0   0.186      -0.120     0.488      0.793


This is a **trade-off curve**, not a ranking. On item-item, $\lambda = 0.3$ lowers the item Gini from about
0.773 to 0.747 for about 0.028 of hit (0.252 to 0.224); $\lambda = 1$ reaches 0.713 at a cost of about
0.14 -- under half the original hit. On the U6 blend the Gini falls at every step too, and the hit
cost is smaller at first ($\lambda = 0.3$: 0.306 to about 0.290). Every point is a legitimate choice;
which one is *right* depends on what a lost hit and a fairer shelf are worth to the people involved -- a
bookshop that wants to sell backlist titles and a reader who wants the bestseller would pick differently.
Notice also that coverage is **not** monotone in $\lambda$ (item-item's peaks in the middle): a heavy
penalty can just swap one small set of shown books for another. That is why we measured Gini, not
coverage alone.

## 3. Feedback loops: what happens after readers read what we show them?

Everything so far was one snapshot. A deployed recommender runs in a **loop**: it shows lists, readers
read some of what they are shown, those reads are logged, and the model is retrained on the new log. The
model's own exposure becomes its next training data.

We simulate that loop. It is a **stylized exposure-to-consumption loop**, not a model of real readers:
each simulated reader reads each book in their top 10 **independently with probability 0.3** (about 3
reads per reader per round), regardless of taste. That is deliberately crude -- real readers are choosier
-- but it isolates the one mechanism we want to see: *what the model shows is what gets read*.

One round, for the 500 cohort readers, in a fixed seeded order:

1. Fit the path on the current train log; serve each reader a top 10 (their seen books excluded).
2. For each served book, in rank order, draw a random number; the reader reads it if the draw is
   below 0.3 -- **unless it is one of that reader's own held-out books** (the *leakage guard*, below).
3. Append every read to the train log as a new positive row (`label = 1`, `session_id = -1`, timestamped
   after the reader's last read) and add it to the reader's seen set.

**Why the leakage guard?** The 500 readers are exactly the readers whose `val` books we score. If the
simulation lets a reader "read" one of their own `val` books, that book lands in the **train** log -- and
the model is then trained on the very answer it is scored against. That is a leak, and it is easy to
write by accident. We do one round by hand, *both* ways, and look at what the leak does.

In [11]:
ACCEPT_PROB = 0.3
train_log = [row for row in rows if row["split"] == "train"]
last_stamp = {}
for row in train_log:
    r = int(row["reader_id"])
    last_stamp[r] = max(last_stamp.get(r, int(row["timestamp"])), int(row["timestamp"]))

rng = np.random.default_rng(SEED)
order = [int(r) for r in rng.choice(cohort, size=len(cohort), replace=False)]   # fixed reader order
served0 = {r: tops["item-item"][r] for r in order}     # round 0 = the scoreboard's item-item lists

guarded, leaky, blocked = [], [], 0
for r in order:
    for item in served0[r]:
        if rng.random() >= ACCEPT_PROB:                # one draw per served slot, in rank order
            continue
        read = {"reader_id": r, "item_id": item, "session_id": -1,
                "timestamp": last_stamp[r] + 1, "split": "train", "label": 1}
        leaky.append(read)                             # an UNGUARDED loop would log every read
        if item in held_out_by[r]:
            blocked += 1                               # the guard: never log a held-out answer
            continue
        last_stamp[r] += 1
        guarded.append(read)

print(f"round 0: {len(order)} readers x {K} slots; {len(leaky)} reads drawn")
print(f"the leakage guard blocked {blocked} reads of a reader's OWN held-out books")
print(f"guarded log gains {len(guarded)} rows; an unguarded log would gain {len(leaky)}")

round 0: 500 readers x 10 slots; 1501 reads drawn
the leakage guard blocked 79 reads of a reader's OWN held-out books
guarded log gains 1422 rows; an unguarded log would gain 1501


Now refit item-item on each grown log and read round 1. Two kinds of readout, kept separate on purpose:

- **Exposure** (coverage, head-share, Gini) is read off what the refit model *serves* -- excluding each
  reader's **grown** seen set, since a book they read last round is not shown again.
- **Clean hit@10** scores the refit model on the **original** `val` books with the **original** train
  seen set -- exactly the Section-1 scoreboard. If we scored against the grown seen set instead, hit
  would move for a bookkeeping reason (newly "seen" books leave the list), not because the model changed.

In [12]:
def round_readout(log_rows, new_reads):
    # Refit item-item on the grown log; exposure of the served lists + clean hit on the original seen.
    path = ItemItemRetrievalPath().fit(log_rows, catalog=catalog_ids)
    seen_now = {r: set(seen_by[r]) for r in order}
    for read in new_reads:
        seen_now[read["reader_id"]].add(read["item_id"])
    served = [[c.item_id for c in rank(path.retrieve(r, {"seen": seen_now[r]}, K), n=K,
                                       exclude=seen_now[r])] for r in order]
    return {"coverage": catalog_coverage(served, catalog_ids), "head_share": head_share(served, head),
            "gini": exposure_gini(served, catalog_ids), "clean_hit": hit_at_10(top10(path))}


round1 = round_readout(train_log + guarded, guarded)
round1_leaky = round_readout(train_log + leaky, leaky)
lists0 = list(served0.values())
print(f"{'':<22} {'coverage':>9} {'head-share':>11} {'Gini':>7} {'clean hit':>10}")
print(f"{'round 0':<22} {catalog_coverage(lists0, catalog_ids):>9.3f} {head_share(lists0, head):>11.3f} "
      f"{exposure_gini(lists0, catalog_ids):>7.3f} {hit_at_10(tops['item-item']):>10.3f}")
for label, res in (("round 1 (guarded)", round1), ("round 1 (UNGUARDED)", round1_leaky)):
    print(f"{label:<22} {res['coverage']:>9.3f} {res['head_share']:>11.3f} {res['gini']:>7.3f} "
          f"{res['clean_hit']:>10.3f}")

                        coverage  head-share    Gini  clean hit
round 0                    0.490       0.491   0.773      0.252
round 1 (guarded)          0.473       0.533   0.802      0.266
round 1 (UNGUARDED)        0.466       0.540   0.806      0.288


Three things to read off that table.

- **Concentration starts in round 1.** One round of reading what it showed, and item-item's coverage drops
  from 0.490 to 0.473 while its Gini rises from 0.773 to 0.802. Books that were shown got read; books that
  got read gain co-reads; books with more co-reads get shown more.
- **The guard blocked 79 reads** -- about 5% of the 1,501 -- that would have put a reader's own held-out
  book into the train log.
- **Those 79 rows are worth +0.022 of hit.** The unguarded model scores 0.288 against the guarded 0.266.
  Nothing about the readers changed; the model was simply trained on some of the answers it is graded
  on. A leaky simulation would report this as "the loop makes the model *better*" -- a common, and
  silent, mistake. Every number below is guarded.

### Reveal: `simulate_feedback_loop`

`simulate_feedback_loop(path_factory, rows, cohort=..., catalog=..., seed=..., rounds=6, k=10,
accept_prob=0.3)` is the round above, repeated: the same single seeded generator (first the reader
order, then one draw per served slot), the same guard, the same separate clean-hit readout. It takes a
**factory** (a class such as `ItemItemRetrievalPath`) because it fits a fresh path every round. Each
round returns a `RoundMetrics` row. Round 0 is the original model; we check that its round 1 is our
hand-made round exactly, then let it run for 6 rounds.

In [13]:
loop = simulate_feedback_loop(ItemItemRetrievalPath, rows, cohort=cohort, catalog=catalog,
                              seed=SEED, rounds=6, k=K, accept_prob=ACCEPT_PROB)
assert loop[0].accepted == len(guarded)
assert [(m["reader_id"], m["item_id"]) for m in loop[0].new_rows] == [
    (g["reader_id"], g["item_id"]) for g in guarded]
for key in ("coverage", "head_share", "gini", "clean_hit"):
    assert abs(getattr(loop[1], key) - round1[key]) < 1e-12, key
print("simulate_feedback_loop round 1 == the by-hand guarded round\n")


def show(name, result):
    print(f"{name}\n{'round':>5} {'coverage':>9} {'head-share':>11} {'Gini':>7} {'genre H':>8} "
          f"{'clean hit':>10} {'reads':>6}")
    for m in result:
        print(f"{m.round:>5} {m.coverage:>9.3f} {m.head_share:>11.3f} {m.gini:>7.3f} "
              f"{m.genre_entropy:>8.3f} {m.clean_hit:>10.3f} {m.accepted:>6}")


show("item-item CF, retrained on its own exposures (p = 0.3, 6 rounds)", loop)
echo = sum(m.accepted for m in loop[:-1])
print(f"\nthe round-6 model trains on {sum(len(s) for s in seen_by.values())} original train positives "
      f"+ {echo} simulated reads of its own recommendations")

simulate_feedback_loop round 1 == the by-hand guarded round

item-item CF, retrained on its own exposures (p = 0.3, 6 rounds)
round  coverage  head-share    Gini  genre H  clean hit  reads
    0     0.490       0.491   0.773    1.640      0.252   1422
    1     0.473       0.533   0.802    1.361      0.266   1425
    2     0.429       0.557   0.828    1.216      0.256   1323
    3     0.412       0.545   0.835    1.184      0.206   1310
    4     0.384       0.522   0.837    1.182      0.154   1373
    5     0.373       0.484   0.833    1.254      0.130   1365
    6     0.358       0.448   0.832    1.317      0.090   1360

the round-6 model trains on 4694 original train positives + 8218 simulated reads of its own recommendations


Over six rounds the loop **concentrates** exposure:

- **Coverage falls from 0.490 to 0.358** -- the shelf of books shown to anyone shrinks by more than a
  quarter. The **Gini rises from 0.773 to 0.832**.
- **Head-share rises and then falls back** (0.491 to 0.557 by round 2, then 0.448). The head is frozen at
  round 0 -- and the loop makes *its own* favourites popular, including books that were tail books at
  round 0. "Rich get richer" here means *the books the model already liked*, not only the old bestsellers.
  That is why we track the Gini, which does not depend on where the head line was drawn.
- **Clean hit collapses: 0.252 to 0.090.** It even ticks up at round 1, then falls every round. By round
  6 the model trains on 4,694 real train positives plus 8,218 simulated reads -- its own echo outnumbers
  the readers almost 2 to 1 -- so item-item is learning which books *it* recommends together, not which
  books *readers* read together.

Keep the simulation's limits in view: every simulated reader reads 30% of what they are shown, whatever
it is. Real readers are choosier, so the *size* of these changes belongs to our made-up reading rule.
The *direction* -- retraining on your own exposures narrows them -- is the lesson.

Does every path do this? Run the same loop on popularity:

In [14]:
pop_loop = simulate_feedback_loop(PopularityRetrievalPath, rows, cohort=cohort, catalog=catalog,
                                  seed=SEED, rounds=6, k=K, accept_prob=ACCEPT_PROB)
show("popularity, retrained on its own exposures", pop_loop)

popularity, retrained on its own exposures
round  coverage  head-share    Gini  genre H  clean hit  reads
    0     0.008       1.000   0.995    2.138      0.108   1470
    1     0.010       1.000   0.993    2.014      0.108   1475
    2     0.012       1.000   0.991    2.102      0.108   1362
    3     0.014       1.000   0.990    2.179      0.108   1379
    4     0.015       1.000   0.990    2.206      0.108   1436
    5     0.018       1.000   0.989    2.155      0.108   1457
    6     0.019       1.000   0.988    2.119      0.108   1437


Popularity hardly moves: its **Gini stays at or above 0.988 and its coverage at or below 0.019 in every
round** (coverage even creeps up from 0.008, because each reader's seen set grows and pushes the next
most popular books into their list). Its clean hit is flat at 0.108. There is nothing left to
concentrate: popularity *starts* where a feedback loop pushes everything else. The loop matters most for
the paths that begin spread out -- like item-item.

### A mitigation: an exploration slot

The simplest standard fix is to **explore**: in every reader's list, replace **slot 10** with a random
book the reader has not seen -- drawn from the warm books (at least one original train read), with the
same seeded generator. Readers then sometimes read something the model would never have shown, and that
read enters the log. (A popularity-penalty re-rank, Section 2's $\lambda = 0.3$, is the loop's other
built-in mitigation: `mitigation="penalty"`.)

What does it cost? Our accuracy readout, clean hit, scores the **retrained model**, so we report two
signed differences against the plain loop: `cost_0` at round 0 and `cost_R` at round 6.

In [15]:
explore = simulate_feedback_loop(ItemItemRetrievalPath, rows, cohort=cohort, catalog=catalog,
                                 seed=SEED, rounds=6, k=K, accept_prob=ACCEPT_PROB,
                                 mitigation="explore")
show("item-item + exploration slot (slot 10 = a random unseen warm book)", explore)
cost_0 = explore[0].clean_hit - loop[0].clean_hit
cost_R = explore[-1].clean_hit - loop[-1].clean_hit
print(f"\nat round 6, exploration vs the plain loop: coverage {explore[-1].coverage - loop[-1].coverage:+.3f}, "
      f"Gini {explore[-1].gini - loop[-1].gini:+.3f}")
print(f"accuracy cost (clean hit, explore - plain): round 0 {cost_0:+.3f}, round 6 {cost_R:+.3f}")

item-item + exploration slot (slot 10 = a random unseen warm book)
round  coverage  head-share    Gini  genre H  clean hit  reads
    0     0.514       0.465   0.751    1.912      0.252   1440
    1     0.495       0.498   0.781    1.717      0.258   1439
    2     0.482       0.523   0.802    1.563      0.250   1319
    3     0.455       0.519   0.812    1.528      0.216   1302
    4     0.441       0.513   0.816    1.545      0.206   1356
    5     0.418       0.481   0.817    1.623      0.190   1398
    6     0.421       0.456   0.812    1.702      0.142   1337

at round 6, exploration vs the plain loop: coverage +0.063, Gini -0.019
accuracy cost (clean hit, explore - plain): round 0 +0.000, round 6 +0.052


- **One random slot already widens the shelf at round 0** (coverage 0.490 to 0.514, Gini 0.773 to
  0.751).
- **At round 6 it partly restores exposure:** coverage 0.421 against 0.358 (+0.063), Gini 0.812 against
  0.832 (-0.019). It *slows* the concentration but does not stop it -- the exploring loop's Gini still
  climbs from 0.751 to 0.812.
- **The retrained model loses nothing, and by round 6 it is better off.** `cost_0` is exactly 0 (both loops
  start from the same log and the same model). `cost_R` is **+0.052**: the exploring loop's model ends at
  0.142 against 0.090, because random reads dilute the echo in its training data.

Be precise about what that does *not* say. Clean hit grades the retrained *model*; it does not grade
the *served list*, where slot 10 is a random book every round. A reader is shown 9 recommendations and
one experiment. Whether that slot is worth it is the same kind of trade-off as the penalty curve -- and
one seed of a stylized loop is evidence about direction, not a production estimate.

## 4. Filter bubbles: does the list show a reader less than they read?

A *filter bubble* is the worry that a recommender shows each reader a **narrower** slice of the world than
they would choose -- and, through the loop, narrows it further. We can measure the first half per
reader, with genres. Each book carries one or more genre labels. The **genre entropy** of a list is the
Shannon entropy (in bits) of its pooled genre labels:

$$H = -\sum_g p_g \log_2 p_g$$

where $p_g$ is the fraction of the list's genre labels that are genre $g$. A list all in one genre has
$H = 0$; spread evenly over 4 genres, $H = 2$ bits. By hand for one reader's item-item list, then the
reveal:

In [16]:
labels = [g for item in tops["item-item"][reader] for g in catalog[item].fields["genres"].split(";")]
p = np.array(list(Counter(labels).values()), dtype=float) / len(labels)
h_by_hand = float(-(p * np.log2(p)).sum())
assert abs(genre_entropy(tops["item-item"][reader], catalog) - h_by_hand) < 1e-12
print(f"reader {reader}: genre labels in the top 10 = {dict(Counter(labels))}")
print(f"genre entropy {h_by_hand:.3f} bits == genre_entropy(...)\n")

spread = genre_spread_vs_history(tops["item-item"], seen_by, catalog)   # {reader: (recs H, history H)}
rec_h, hist_h = (np.array(v) for v in zip(*spread.values()))
print(f"item-item, mean over {len(spread)} readers: recommendations {rec_h.mean():.3f} bits, "
      f"own history {hist_h.mean():.3f} bits")
print(f"readers whose top 10 is narrower than their history: {np.mean(rec_h < hist_h):.1%}\n")
print(f"{'path':<14} {'genre H of the top 10':>22} {'readers narrower than history':>30}")
for name, lists in tops.items():
    rec_h = np.array([h for h, _ in genre_spread_vs_history(lists, seen_by, catalog).values()])
    print(f"{name:<14} {rec_h.mean():>22.3f} {np.mean(rec_h < hist_h):>30.1%}")

reader 1: genre labels in the top 10 = {'adventure': 4, 'sci-fi': 1, 'children': 3, 'horror': 1, 'philosophy': 2}
genre entropy 2.118 bits == genre_entropy(...)

item-item, mean over 500 readers: recommendations 1.640 bits, own history 1.564 bits
readers whose top 10 is narrower than their history: 42.0%

path            genre H of the top 10  readers narrower than history
popularity                      2.138                          27.0%
lexical                         0.300                          89.0%
semantic                        1.927                          24.4%
item-item                       1.640                          42.0%
MF                              1.375                          58.2%
two-tower                       0.681                          83.6%
feature-tower                   0.348                          89.6%
U10 hybrid                      0.513                          87.2%
U6 blend                        1.385                          54.0%


Reading it in three steps:

- **Item-item is not a bubble on average.** Its lists (1.640 bits) are about as wide as readers' own
  histories (1.564 bits); 42% of readers get a narrower list, 58% a wider one.
- **The most accurate paths are the narrowest.** The two-tower (0.681 bits), the U10 hybrid (0.513) and
  the feature tower (0.348) give 84-90% of readers a list narrower than what they actually read; lexical
  (0.300) is just as narrow, because keyword matches stay inside a genre. Popularity and semantic are the
  widest -- popularity because it ignores the reader entirely. Precise "more of what you read" is often
  exactly what hits a `val` book, so narrowness and accuracy travel together here too.
- **The loop narrows, then partly rebounds.** Look back at the `genre H` column of Section 3: item-item's
  lists fall from 1.640 bits to **1.182 at round 4** (-0.46 bits, below the readers' own 1.564) and then
  climb back to 1.317 by round 6. In this simulation the narrowing is real but **transient**: once a reader
  has read the unseen books of their favourite genres (seen books are never shown again), the list has to
  move on. The exploration slot keeps lists wider throughout (its lowest is 1.528).

One caution about the comparison with history: a 10-book list is a small sample, and a small sample tends
to show fewer distinct labels than a long history. So "narrower than history" is a rough reading; the
cleaner comparison is **between paths at the same k**, which is the table above.

## 5. What offline metrics cannot see

### Offline scores reflect the logging policy

Our `val` books are what readers *actually read next* -- in a world where they found books through
whatever exposure produced the log. So the targets themselves lean popular, and a path that leans popular
is partly rewarded for *agreeing with the past exposure*, not only for knowing the reader. We can measure
how strong that association is: what share of the val targets are head books, and how well each path hits
**head** targets versus **tail** targets.

In [17]:
targets = [i for r in cohort for i in val_by[r] - seen_by[r]]
print(f"{len(targets)} val targets; {np.mean([i in head for i in targets]):.1%} are head books "
      f"(the head is {len(head) / len(catalog_ids):.0%} of the catalog)\n")
print(f"{'path':<12} {'hit on head targets':>20} {'hit on tail targets':>20}")
for name in ("item-item", "MF", "two-tower", "U6 blend"):
    head_hits, tail_hits = [], []
    for r in cohort:
        relevant, top = val_by[r] - seen_by[r], set(tops[name][r])
        if relevant & head:
            head_hits.append(float(bool(top & relevant & head)))
        if relevant - head:
            tail_hits.append(float(bool(top & (relevant - head))))
    print(f"{name:<12} {np.mean(head_hits):>20.3f} {np.mean(tail_hits):>20.3f}")

1082 val targets; 44.1% are head books (the head is 10% of the catalog)

path          hit on head targets  hit on tail targets
item-item                   0.356                0.032
MF                          0.427                0.000
two-tower                   0.517                0.014


U6 blend                    0.446                0.024


**44.1% of the val targets are head books**, although the head is only 10% of the catalog. And every
path is far better at hitting a head target than a tail target: item-item 0.356 vs 0.032 (about 11x),
the U6 blend 0.446 vs 0.024, the two-tower 0.517 vs 0.014, and MF never hits a tail target at all.

This is a **measured association, not a causal claim**. From the log alone we cannot tell how much of it
is readers genuinely preferring head books and how much is the past exposure that made head books the
ones readers found. What we *can* say is that an offline score computed on this log rewards agreeing with
it. So read Section 2's curve with care: its hit cost is measured against these head-heavy targets. A
re-rank that shows more tail books loses offline hit partly because the log's targets are mostly head
books. Only an **online** test -- showing the lists to real readers -- would settle what readers do with
them. An offline metric is a strong, cheap filter; it is not the final word.

### Closing the cold-start thread

The cold-start thread started in Unit 6 (a reader with no history: popularity carries the blend) and
continued in Unit 9 (a book with no reads: the feature tower). Here is its last readout. Of the catalog's
books, **818 have zero train reads**. How many of them does each path ever put in anyone's top 10?

In [18]:
train_items = {i for items in seen_by.values() for i in items}
zero_train = set(catalog_ids) - train_items       # books with NO train reads at all
print(f"{len(zero_train)} catalog books have zero train reads\n")
print(f"{'path':<14} {'zero-train books surfaced (any top 10)':>40}")
for name, lists in tops.items():
    surfaced = {i for top in lists.values() for i in top} & zero_train
    print(f"{name:<14} {len(surfaced):>40}")

818 catalog books have zero train reads

path             zero-train books surfaced (any top 10)
popularity                                            0
lexical                                             128
semantic                                            263
item-item                                             0
MF                                                    0
two-tower                                             0
feature-tower                                       109
U10 hybrid                                           17
U6 blend                                              4


- The **warm-only collaborative paths -- popularity, item-item, MF, two-tower -- surface exactly 0** of the
  818. A book nobody read in train has no collaborative signal at all, so it can never rise to the top.
- The **feature tower surfaces 109**: it is the only *learned, collaborative* path that serves cold
  books, because it scores a book from its features (Unit 9).
- The **content paths surface more**: lexical 128, semantic 263 -- they never needed reads in the first
  place. The composites inherit their content members' reach: the U10 hybrid surfaces 17 (through its
  BM25 side), the U6 blend 4 (through lexical).

Seen through this unit's lens, cold start is also an **exposure** question: the most accurate paths give
a brand-new book -- and its author -- zero exposure, and the feedback loop would keep it that way, since
a book that is never shown is never read.

## Where this goes

- **Accuracy is one column.** On the same 500 readers, the most accurate paths are the most concentrated:
  the U10 hybrid (hit 0.362) gives 86% of its slots to the 200 head books and has an item Gini of 0.922;
  item-item (0.252) covers 49% of the catalog at a Gini of 0.773.
- **Exposure fairness** is measured with descriptive proxies: the item Gini over the full catalog, the
  long-tail share, and the author Gini read against the catalog's own 0.211. A popularity-penalty re-rank
  draws an accuracy/exposure **trade-off curve** (item-item: Gini 0.773 to 0.713, hit 0.252 to 0.112)
  -- a menu of choices, not a winner.
- **Feedback loops** concentrate exposure: retrained on its own (guarded) exposures for 6 rounds,
  item-item's coverage falls from 0.490 to 0.358 and its Gini rises to 0.832. An exploration slot slows
  that (coverage +0.063, Gini -0.019 at round 6). A loop that leaks held-out answers into its train log
  reports a fake accuracy *gain*.
- **Filter bubbles** are measurable as genre spread: the accurate neural paths show far narrower lists
  than readers' histories, and the loop narrows item-item's lists for a few rounds before a partial
  rebound.
- **Offline metrics inherit the log**: 44% of val targets are head books, and each collaborative path we
  checked hits head targets at least 10 times as often as tail targets. **Cold books** get zero exposure from
  every warm-only collaborative path; only the feature tower (and the content paths) serve them.

### The bridge ahead

**Checkpoint B** is Part 2's assessment. You will score the semantic path, the two-tower and the hybrid;
redo the cold-start count; compare recency and transitions with item-item on the session log; measure a
blend's Gini, long-tail share and genre spread and run two short feedback loops; and then build a
Part-2 blend on `val`, **freeze** it, and unseal the sealed final split **once** -- as in Checkpoint A.
**Unit 14** is the **capstone**: the full multi-path recommender, benchmarked against the Part-1 build,
with an ablation table *and* this unit's beyond-accuracy table beside its hit@10.